In [3]:
from pathlib import Path
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

from ewatercycle.models import Wflow
from ewatercycle.parameter_sets import available_parameter_sets
from ewatercycle.forcing import sources

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", None)

/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498


In [4]:
def scores(comparison):
    obs = comparison["Q_grdc_m3s"]
    sim = comparison["Q_model_m3s"]

    #algemene scores
    bias = (sim - obs).mean()
    rmse = np.sqrt(((sim - obs) ** 2).mean())

    #log-NSE
    log_obs = np.log(obs)
    log_sim = np.log(sim)

    log_nse = 1 - ((log_sim - log_obs) ** 2).sum() / ((log_obs - log_obs.mean()) **2).sum()

    #low_flow selectie
    low_1600 = obs < 1600

    #dagen onder drempels
    grdc_days_below_1600 = (obs < 1600).sum()
    model_days_below_1600 = (sim < 1600).sum()

    grdc_days_below_1020 = (obs < 1020).sum()
    model_days_below_1020 = (sim < 1020).sum()

    #percentielen
    grdc_p10 = obs.quantile(0.10)
    model_p10 = sim.quantile(0.10)

    grdc_p17 = obs.quantile(0.17)
    model_p17 = sim.quantile(0.17)

    scores = {
        "bias": bias,
        "rmse": rmse,
        "log_nse": log_nse,

        "bias_low_1600": (sim[low_1600] - obs[low_1600]).mean(),
        "mae_low_1600": abs(sim[low_1600] - obs[low_1600]).mean(),

        "grdc_days_below_1600": grdc_days_below_1600,
        "model_days_below_1600": model_days_below_1600,
        "diff_days_below_1600": model_days_below_1600 - grdc_days_below_1600,

        "grdc_days_below_1020": grdc_days_below_1020,
        "model_days_below_1020": model_days_below_1020,
        "diff_days_below_1020": model_days_below_1020 - grdc_days_below_1020,

        "grdc_p10": grdc_p10,
        "model_p10": model_p10,
        "diff_p10": model_p10 - grdc_p10,

        "grdc_p17": grdc_p17,
        "model_p17": model_p17,
        "diff_p17": model_p17 - grdc_p17,
    }

    return scores

In [5]:
project_dir = Path("/home/niels/BEP-Niels")

temp_dir = project_dir / "temp_runs"
results_dir = project_dir / "results"
figures_dir = project_dir / "Results-figures"

forcing_dir = temp_dir / "forcing_basisrun_1987_1995"

model_start = "1986-01-01T00:00:00Z"
model_end = "1996-01-01T00:00:00Z"

calibration_start = "1987-01-01"
calibration_end = "1995-12-31"

print("Projectmap bestaat:", project_dir.exists())
print("Forcingmap bestaat:", forcing_dir.exists())
print("Resultsmap bestaat:", results_dir.exists())

Projectmap bestaat: True
Forcingmap bestaat: True
Resultsmap bestaat: True


In [6]:
grdc_file = project_dir / "Data Lobith" / "data grdc" / "6435060_Q_Day.Cmd.txt"

grdc = pd.read_csv(
    grdc_file,
    sep=";",
    comment="#",
    skipinitialspace=True,
    na_values=-999.000,
    encoding="latin1"
)

grdc.columns = grdc.columns.str.strip()

grdc = grdc.rename(columns={
    "YYYY-MM-DD": "date",
    "Value": "Q_grdc_m3s"
})

grdc["date"] = pd.to_datetime(grdc["date"])
grdc["Q_grdc_m3s"] = pd.to_numeric(grdc["Q_grdc_m3s"])

grdc = grdc.dropna(subset=["Q_grdc_m3s"])

grdc_calibration = grdc[
    (grdc["date"] >= calibration_start) &
    (grdc["date"] <= calibration_end)
].copy()

grdc_calibration = grdc_calibration[["date", "Q_grdc_m3s"]]

print("Aantal dagen:", len(grdc_calibration))
print("Start:", grdc_calibration["date"].min())
print("Einde:", grdc_calibration["date"].max())

grdc_calibration.head()

Aantal dagen: 3287
Start: 1987-01-01 00:00:00
Einde: 1995-12-31 00:00:00


,date,Q_grdc_m3s
31411,1987-01-01,4865.0
31412,1987-01-02,5756.0
31413,1987-01-03,6160.0
31414,1987-01-04,6973.0
31415,1987-01-05,7579.0


In [41]:
nc_files = sorted(forcing_dir.rglob("*.nc"))

print("Aantal .nc bestanden gevonden:", len(nc_files))

for i, file in enumerate(nc_files):
    print(i, file)

Aantal .nc bestanden gevonden: 7
0 /home/niels/BEP-Niels/temp_runs/forcing_basisrun_1987_1995/preproc/diagnostic/orog/OBS6_ERA5_reanaly_1_fx_orog.nc
1 /home/niels/BEP-Niels/temp_runs/forcing_basisrun_1987_1995/preproc/diagnostic/pr/OBS6_ERA5_reanaly_1_day_pr_1986-1995.nc
2 /home/niels/BEP-Niels/temp_runs/forcing_basisrun_1987_1995/preproc/diagnostic/psl/OBS6_ERA5_reanaly_1_day_psl_1986-1995.nc
3 /home/niels/BEP-Niels/temp_runs/forcing_basisrun_1987_1995/preproc/diagnostic/rsds/OBS6_ERA5_reanaly_1_day_rsds_1986-1995.nc
4 /home/niels/BEP-Niels/temp_runs/forcing_basisrun_1987_1995/preproc/diagnostic/rsdt/OBS6_ERA5_reanaly_1_CFday_rsdt_1986-1995.nc
5 /home/niels/BEP-Niels/temp_runs/forcing_basisrun_1987_1995/preproc/diagnostic/tas/OBS6_ERA5_reanaly_1_day_tas_1986-1995.nc
6 /home/niels/BEP-Niels/temp_runs/forcing_basisrun_1987_1995/work/diagnostic/script/wflow_ERA5_Rhine_1986_1995.nc


In [42]:
forcing_file = nc_files[-1]

print("Gekozen forcingbestand:")
print(forcing_file)

ds = xr.open_dataset(forcing_file)

print("\nVariabelen in dit bestand:")
print(list(ds.data_vars))

ds.close()

Gekozen forcingbestand:
/home/niels/BEP-Niels/temp_runs/forcing_basisrun_1987_1995/work/diagnostic/script/wflow_ERA5_Rhine_1986_1995.nc

Variabelen in dit bestand:
['pr', 'time_bnds', 'lat_bnds', 'lon_bnds', 'tas', 'pet']


In [43]:
WflowForcing = sources["WflowForcing"]

forcing = WflowForcing(
    start_time=model_start,
    end_time=model_end,
    directory=str(forcing_file.parent),
    netcdfinput=forcing_file.name,
    Precipitation="/pr",
    EvapoTranspiration="/pet",
    Temperature="/tas",
    Inflow=None,
)

print("Bestaande forcing is gekoppeld")

Bestaande forcing is gekoppeld


In [44]:
parameter_sets = available_parameter_sets(target_model="wflow")
parameter_set = parameter_sets["wflow_rhine_sbm_nc"]

parameter_set.config = Path(parameter_set.directory) / "wflow_sbm_NC.ini"

lat_lobith = 51.849999998
lon_lobith = 6.0999999998

print("Parameter set:", parameter_set.directory)
print("Config bestaat:", parameter_set.config.exists())
print("Lobith:", lat_lobith, lon_lobith)

Parameter set: /data/shared/parameter-sets/wflow_rhine_sbm_nc
Config bestaat: True
Lobith: 51.849999998 6.0999999998


In [45]:
basis_file = results_dir / "basisrun_1987_1995_lobith_daily.csv"

print("Basisrun-output bestaat:", basis_file.exists())
print(basis_file)

Basisrun-output bestaat: True
/home/niels/BEP-Niels/results/basisrun_1987_1995_lobith_daily.csv


In [46]:
model_basis = pd.read_csv(basis_file)
model_basis["date"] = pd.to_datetime(model_basis["date"])

model_basis.head()

,date,Q_model_m3s
0,1987-01-01,13869.678711
1,1987-01-02,15741.568359
2,1987-01-03,16061.675781
3,1987-01-04,15124.300781
4,1987-01-05,13978.630859


In [47]:
comparison_basis = pd.merge(
    grdc_calibration,
    model_basis,
    on="date",
    how="inner"
)

print("Aantal overlappende dagen:", len(comparison_basis))

comparison_basis.head()

Aantal overlappende dagen: 1826


,date,Q_grdc_m3s,Q_model_m3s
0,1987-01-01,4865.0,13869.678711
1,1987-01-02,5756.0,15741.568359
2,1987-01-03,6160.0,16061.675781
3,1987-01-04,6973.0,15124.300781
4,1987-01-05,7579.0,13978.630859


In [48]:
basis_scores = scores(comparison_basis)

pd.DataFrame([basis_scores]).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
0,581.95,1876.37,-1.04,-146.99,595.2,669,760,91,96,485,389,1131.0,496.45,-634.55,1271.0,735.06,-535.94


In [49]:
def multiply_tbl(tbl_file, factor):
    lines = Path(tbl_file).read_text().splitlines()
    new_lines = []

    for line in lines:
        parts = line.split()

        if len(parts) == 0:
            new_lines.append(line)
            continue

        try:
            value = float(parts[-1])
            parts[-1] = str(value * factor)
            new_lines.append(" ".join(parts))
        except:
            new_lines.append(line)

    Path(tbl_file).write_text("\n".join(new_lines))

In [50]:
run_name = "sens_RootingDepth_x0p75"

parameter_name = "RootingDepth.tbl"
factor = 0.75

run_dir = temp_dir / run_name

print("Run:", run_name)
print("Parameter:", parameter_name)
print("Factor:", factor)
print("Runmap:", run_dir)

Run: sens_RootingDepth_x0p75
Parameter: RootingDepth.tbl
Factor: 0.75
Runmap: /home/niels/BEP-Niels/temp_runs/sens_RootingDepth_x0p75


In [51]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))

cfg_dir = Path(cfg_dir)

tbl_files = list(cfg_dir.rglob(parameter_name))

print("Aantal gevonden parameterbestanden:", len(tbl_files))

for tbl_file in tbl_files:
    print("Aanpassen:", tbl_file)
    multiply_tbl(tbl_file, factor)

Aantal gevonden parameterbestanden: 1
Aanpassen: /home/niels/BEP-Niels/temp_runs/sens_RootingDepth_x0p75/intbl/RootingDepth.tbl


In [52]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Run klaar")    

365 1986-12-31 00:00:00 11877.4
730 1987-12-31 00:00:00 2224.1
1095 1988-12-30 00:00:00 3990.0
1460 1989-12-30 00:00:00 4737.5
1825 1990-12-30 00:00:00 6614.1
2190 1991-12-30 00:00:00 7365.6
2555 1992-12-29 00:00:00 1836.9
2920 1993-12-29 00:00:00 10872.5
3285 1994-12-29 00:00:00 5405.4
3650 1995-12-29 00:00:00 10603.7
Run klaar


In [53]:
model_sens = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_sens["date"] = model_sens["date"].dt.tz_convert(None).dt.floor("D")

model_sens = model_sens[
    (model_sens["date"] >= calibration_start) &
    (model_sens["date"] <= calibration_end)
].copy()

output_file = results_dir / f"{run_name}_lobith_daily.csv"

model_sens.to_csv(output_file, index=False)

print("Aantal dagen:", len(model_sens))
print("Opgeslagen als:", output_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/sens_RootingDepth_x0p75_lobith_daily.csv


In [54]:
comparison_sens = pd.merge(
    grdc_calibration,
    model_sens,
    on="date",
    how="inner"
)

sens_scores = scores(comparison_sens)

pd.DataFrame([basis_scores, sens_scores], index=["basis", run_name]).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis,581.95,1876.37,-1.04,-146.99,595.20,669,760,91,96,485,389,1131.0,496.45,-634.55,1271.00,735.06,-535.94
sens_RootingDepth_x0p75,731.00,2024.48,-0.95,-103.10,581.15,1150,1281,131,125,804,679,1187.2,577.05,-610.15,1310.62,784.62,-526.00


In [55]:
print("Basis:")
print("aantal rijen:", len(model_basis))
print("unieke datums:", model_basis["date"].nunique())

print("\nSensitivity:")
print("aantal rijen:", len(model_sens))
print("unieke datums:", model_sens["date"].nunique())

print("\nDubbele datums in sensitivity:")
print(model_sens["date"].duplicated().sum())

Basis:
aantal rijen: 1826
unieke datums: 1826

Sensitivity:
aantal rijen: 3287
unieke datums: 3287

Dubbele datums in sensitivity:
0


In [56]:
print("GRDC:")
print(grdc_calibration["date"].min(), grdc_calibration["date"].max(), len(grdc_calibration))

print("\nBasisrun:")
print(model_basis["date"].min(), model_basis["date"].max(), len(model_basis))

print("\nSensitivity:")
print(model_sens["date"].min(), model_sens["date"].max(), len(model_sens))

GRDC:
1987-01-01 00:00:00 1995-12-31 00:00:00 3287

Basisrun:
1987-01-01 00:00:00 1991-12-31 00:00:00 1826

Sensitivity:
1987-01-01 00:00:00 1995-12-31 00:00:00 3287


In [57]:
run_name = "basisrun_full_1987_1995"
run_dir = temp_dir / run_name

print("Run:", run_name)
print("Runmap:", run_dir)

Run: basisrun_full_1987_1995
Runmap: /home/niels/BEP-Niels/temp_runs/basisrun_full_1987_1995


In [58]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))

print("Basisrun is voorbereid")
print(cfg_file)

Basisrun is voorbereid
/home/niels/BEP-Niels/temp_runs/basisrun_full_1987_1995/wflow_ewatercycle.ini


In [59]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Basisrun klaar")

365 1986-12-31 00:00:00 11876.8
730 1987-12-31 00:00:00 2224.0
1095 1988-12-30 00:00:00 3958.7
1460 1989-12-30 00:00:00 4676.8
1825 1990-12-30 00:00:00 6410.8
2190 1991-12-30 00:00:00 7189.5
2555 1992-12-29 00:00:00 1827.6
2920 1993-12-29 00:00:00 10864.1
3285 1994-12-29 00:00:00 5357.5
3650 1995-12-29 00:00:00 10540.7
Basisrun klaar


In [60]:
model_basis_full = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_basis_full["date"] = model_basis_full["date"].dt.tz_convert(None).dt.floor("D")

model_basis_full = model_basis_full[
    (model_basis_full["date"] >= calibration_start) &
    (model_basis_full["date"] <= calibration_end)
].copy()

basis_full_file = results_dir / "basisrun_full_1987_1995_lobith_daily.csv"

model_basis_full.to_csv(basis_full_file, index=False)

print("Aantal dagen:", len(model_basis_full))
print("Opgeslagen als:", basis_full_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/basisrun_full_1987_1995_lobith_daily.csv


In [61]:
comparison_basis_full = pd.merge(
    grdc_calibration,
    model_basis_full,
    on="date",
    how="inner"
)

comparison_sens = pd.merge(
    grdc_calibration,
    model_sens,
    on="date",
    how="inner"
)

basis_scores_full = scores(comparison_basis_full)
sens_scores = scores(comparison_sens)

pd.DataFrame(
    [basis_scores_full, sens_scores],
    index=["basis_full", "sens_RootingDepth_x0p75"]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_RootingDepth_x0p75,731.0,2024.48,-0.95,-103.10,581.15,1150,1281,131,125,804,679,1187.2,577.05,-610.15,1310.62,784.62,-526.00


In [62]:
parameter_name = "FirstZoneCapacity.tbl"
factor = 1.25
run_name = "sens_FirstZoneCapacity_x1p25"

In [63]:
run_name = "sens_FirstZoneCapacity_x1p25"

parameter_name = "FirstZoneCapacity.tbl"
factor = 1.25

run_dir = temp_dir / run_name

print("Run:", run_name)
print("Parameter:", parameter_name)
print("Factor:", factor)
print("Runmap:", run_dir)

Run: sens_FirstZoneCapacity_x1p25
Parameter: FirstZoneCapacity.tbl
Factor: 1.25
Runmap: /home/niels/BEP-Niels/temp_runs/sens_FirstZoneCapacity_x1p25


In [64]:
pd.DataFrame(
    [basis_scores_full, sens_scores],
    index=["basis_full", run_name]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_FirstZoneCapacity_x1p25,731.0,2024.48,-0.95,-103.10,581.15,1150,1281,131,125,804,679,1187.2,577.05,-610.15,1310.62,784.62,-526.00


In [65]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))
cfg_dir = Path(cfg_dir)

tbl_files = list(cfg_dir.rglob(parameter_name))

print("Aantal gevonden parameterbestanden:", len(tbl_files))

for tbl_file in tbl_files:
    print("Aanpassen:", tbl_file)
    multiply_tbl(tbl_file, factor)

Aantal gevonden parameterbestanden: 1
Aanpassen: /home/niels/BEP-Niels/temp_runs/sens_FirstZoneCapacity_x1p25/intbl/FirstZoneCapacity.tbl


In [66]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Run klaar")

365 1986-12-31 00:00:00 11876.8
730 1987-12-31 00:00:00 2224.0
1095 1988-12-30 00:00:00 3958.7
1460 1989-12-30 00:00:00 4676.8
1825 1990-12-30 00:00:00 6410.8
2190 1991-12-30 00:00:00 7189.5
2555 1992-12-29 00:00:00 1827.6
2920 1993-12-29 00:00:00 10864.1
3285 1994-12-29 00:00:00 5357.5
3650 1995-12-29 00:00:00 10540.7
Run klaar


In [67]:
model_FirstZoneCapacity = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_FirstZoneCapacity["date"] = model_FirstZoneCapacity["date"].dt.tz_convert(None).dt.floor("D")

model_FirstZoneCapacity = model_FirstZoneCapacity[
    (model_FirstZoneCapacity["date"] >= calibration_start) &
    (model_FirstZoneCapacity["date"] <= calibration_end)
].copy()

output_file = results_dir / f"{run_name}_lobith_daily.csv"

model_FirstZoneCapacity.to_csv(output_file, index=False)

print("Aantal dagen:", len(model_FirstZoneCapacity))
print("Opgeslagen als:", output_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/sens_FirstZoneCapacity_x1p25_lobith_daily.csv


In [68]:
comparison_FirstZoneCapacity = pd.merge(
    grdc_calibration,
    model_FirstZoneCapacity,
    on="date",
    how="inner"
)

scores_FirstZoneCapacity = scores(comparison_FirstZoneCapacity)

pd.DataFrame(
    [basis_scores_full, scores_FirstZoneCapacity],
    index=["basis_full", run_name]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_FirstZoneCapacity_x1p25,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38


In [69]:
pd.DataFrame(
    [basis_scores_full, sens_scores, scores_FirstZoneCapacity],
    index=[
        "basis_full",
        "sens_RootingDepth_x0p75",
        "sens_FirstZoneCapacity_x1p25"
    ]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_RootingDepth_x0p75,731.0,2024.48,-0.95,-103.10,581.15,1150,1281,131,125,804,679,1187.2,577.05,-610.15,1310.62,784.62,-526.00
sens_FirstZoneCapacity_x1p25,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38


In [70]:
check = pd.merge(
    model_basis_full,
    model_FirstZoneCapacity,
    on="date",
    suffixes=("_basis", "_firstzone")
)

check["verschil"] = check["Q_model_m3s_firstzone"] - check["Q_model_m3s_basis"]

print("Max verschil:", check["verschil"].abs().max())
print("Gemiddeld verschil:", check["verschil"].mean())
check.head()

Max verschil: 0.0
Gemiddeld verschil: 0.0


,date,Q_model_m3s_basis,Q_model_m3s_firstzone,verschil
0,1987-01-01,13869.678711,13869.678711,0.0
1,1987-01-02,15741.568359,15741.568359,0.0
2,1987-01-03,16061.675781,16061.675781,0.0
3,1987-01-04,15124.300781,15124.300781,0.0
4,1987-01-05,13978.630859,13978.630859,0.0


In [71]:
tbl_files = list(run_dir.rglob("FirstZoneCapacity.tbl"))

print("Aantal gevonden bestanden:", len(tbl_files))

for file in tbl_files:
    print(file)

Aantal gevonden bestanden: 2
/home/niels/BEP-Niels/temp_runs/sens_FirstZoneCapacity_x1p25/intbl/FirstZoneCapacity.tbl
/home/niels/BEP-Niels/temp_runs/sens_FirstZoneCapacity_x1p25/run_default/intbl/FirstZoneCapacity.tbl


In [72]:
tbl_file = tbl_files[0]

lines = Path(tbl_file).read_text().splitlines()

for line in lines[:15]:
    print(line)

1 1 1 500.0
2 1 1 8875.0
3 1 1 8875.0
4 1 1 8875.0
5 1 1 8875.0
6 1 1 8875.0
1 2 1 4000.0
2 2 1 12500.0
3 2 1 12500.0
4 2 1 12500.0
5 2 1 12500.0
6 2 1 12500.0
1 3 1 500.0
2 3 1 1250.0
3 3 1 1250.0


In [74]:
run_name = "sens_FirstZoneKsatVer_x0p75"

parameter_name = "FirstZoneKsatVer.tbl"
factor = 0.75

run_dir = temp_dir / run_name

print("Run:", run_name)
print("Parameter:", parameter_name)
print("Factor:", factor)
print("Runmap:", run_dir)

Run: sens_FirstZoneKsatVer_x0p75
Parameter: FirstZoneKsatVer.tbl
Factor: 0.75
Runmap: /home/niels/BEP-Niels/temp_runs/sens_FirstZoneKsatVer_x0p75


In [75]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))
cfg_dir = Path(cfg_dir)

tbl_files = list(cfg_dir.rglob(parameter_name))

print("Aantal gevonden parameterbestanden:", len(tbl_files))

for tbl_file in tbl_files:
    print("Aanpassen:", tbl_file)
    multiply_tbl(tbl_file, factor)

Aantal gevonden parameterbestanden: 1
Aanpassen: /home/niels/BEP-Niels/temp_runs/sens_FirstZoneKsatVer_x0p75/intbl/FirstZoneKsatVer.tbl


In [76]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Run klaar")

365 1986-12-31 00:00:00 11876.8
730 1987-12-31 00:00:00 2224.0
1095 1988-12-30 00:00:00 3958.7
1460 1989-12-30 00:00:00 4676.8
1825 1990-12-30 00:00:00 6410.8
2190 1991-12-30 00:00:00 7189.5
2555 1992-12-29 00:00:00 1827.6
2920 1993-12-29 00:00:00 10864.1
3285 1994-12-29 00:00:00 5357.5
3650 1995-12-29 00:00:00 10540.7
Run klaar


In [77]:
model_FirstZoneKsatVer = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_FirstZoneKsatVer["date"] = model_FirstZoneKsatVer["date"].dt.tz_convert(None).dt.floor("D")

model_FirstZoneKsatVer = model_FirstZoneKsatVer[
    (model_FirstZoneKsatVer["date"] >= calibration_start) &
    (model_FirstZoneKsatVer["date"] <= calibration_end)
].copy()

output_file = results_dir / f"{run_name}_lobith_daily.csv"

model_FirstZoneKsatVer.to_csv(output_file, index=False)

print("Aantal dagen:", len(model_FirstZoneKsatVer))
print("Opgeslagen als:", output_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/sens_FirstZoneKsatVer_x0p75_lobith_daily.csv


In [78]:
comparison_FirstZoneKsatVer = pd.merge(
    grdc_calibration,
    model_FirstZoneKsatVer,
    on="date",
    how="inner"
)

scores_FirstZoneKsatVer = scores(comparison_FirstZoneKsatVer)

pd.DataFrame(
    [basis_scores_full, scores_FirstZoneKsatVer],
    index=["basis_full", run_name]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_FirstZoneKsatVer_x0p75,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38


In [79]:
check = pd.merge(
    model_basis_full,
    model_FirstZoneKsatVer,
    on="date",
    suffixes=("_basis", "_ksat")
)

check["verschil"] = check["Q_model_m3s_ksat"] - check["Q_model_m3s_basis"]

print("Max verschil:", check["verschil"].abs().max())
print("Gemiddeld verschil:", check["verschil"].mean())

Max verschil: 0.0
Gemiddeld verschil: 0.0


In [80]:
run_name = "sens_MaxCanopyStorage_x0p75"

parameter_name = "MaxCanopyStorage.tbl"
factor = 0.75

run_dir = temp_dir / run_name

print("Run:", run_name)
print("Parameter:", parameter_name)
print("Factor:", factor)
print("Runmap:", run_dir)

Run: sens_MaxCanopyStorage_x0p75
Parameter: MaxCanopyStorage.tbl
Factor: 0.75
Runmap: /home/niels/BEP-Niels/temp_runs/sens_MaxCanopyStorage_x0p75


In [81]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))
cfg_dir = Path(cfg_dir)

tbl_files = list(cfg_dir.rglob(parameter_name))

print("Aantal gevonden parameterbestanden:", len(tbl_files))

for tbl_file in tbl_files:
    print("Aanpassen:", tbl_file)
    multiply_tbl(tbl_file, factor)

Aantal gevonden parameterbestanden: 1
Aanpassen: /home/niels/BEP-Niels/temp_runs/sens_MaxCanopyStorage_x0p75/intbl/MaxCanopyStorage.tbl


In [82]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Run klaar")

365 1986-12-31 00:00:00 11882.7
730 1987-12-31 00:00:00 2226.3
1095 1988-12-30 00:00:00 3961.0
1460 1989-12-30 00:00:00 4679.0
1825 1990-12-30 00:00:00 6419.0
2190 1991-12-30 00:00:00 7190.0
2555 1992-12-29 00:00:00 1829.1
2920 1993-12-29 00:00:00 10868.7
3285 1994-12-29 00:00:00 5359.6
3650 1995-12-29 00:00:00 10546.7
Run klaar


In [83]:
model_MaxCanopyStorage = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_MaxCanopyStorage["date"] = model_MaxCanopyStorage["date"].dt.tz_convert(None).dt.floor("D")

model_MaxCanopyStorage = model_MaxCanopyStorage[
    (model_MaxCanopyStorage["date"] >= calibration_start) &
    (model_MaxCanopyStorage["date"] <= calibration_end)
].copy()

output_file = results_dir / f"{run_name}_lobith_daily.csv"

model_MaxCanopyStorage.to_csv(output_file, index=False)

print("Aantal dagen:", len(model_MaxCanopyStorage))
print("Opgeslagen als:", output_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/sens_MaxCanopyStorage_x0p75_lobith_daily.csv


In [84]:
comparison_MaxCanopyStorage = pd.merge(
    grdc_calibration,
    model_MaxCanopyStorage,
    on="date",
    how="inner"
)

scores_MaxCanopyStorage = scores(comparison_MaxCanopyStorage)

pd.DataFrame(
    [basis_scores_full, scores_MaxCanopyStorage],
    index=["basis_full", run_name]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_MaxCanopyStorage_x0p75,721.4,2005.52,-0.93,-108.78,575.95,1150,1284,134,125,804,679,1187.2,576.41,-610.79,1310.62,783.59,-527.03


In [85]:
check = pd.merge(
    model_basis_full,
    model_MaxCanopyStorage,
    on="date",
    suffixes=("_basis", "_canopy")
)

check["verschil"] = check["Q_model_m3s_canopy"] - check["Q_model_m3s_basis"]

print("Max verschil:", check["verschil"].abs().max())
print("Gemiddeld verschil:", check["verschil"].mean())

Max verschil: 72.298828125
Gemiddeld verschil: 11.306983413905186


In [86]:
run_name = "sens_InfiltCapSoil_x1p25"

parameter_name = "InfiltCapSoil.tbl"
factor = 1.25

run_dir = temp_dir / run_name

print("Run:", run_name)
print("Parameter:", parameter_name)
print("Factor:", factor)
print("Runmap:", run_dir)

Run: sens_InfiltCapSoil_x1p25
Parameter: InfiltCapSoil.tbl
Factor: 1.25
Runmap: /home/niels/BEP-Niels/temp_runs/sens_InfiltCapSoil_x1p25


In [87]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))
cfg_dir = Path(cfg_dir)

tbl_files = list(cfg_dir.rglob(parameter_name))

print("Aantal gevonden parameterbestanden:", len(tbl_files))

for tbl_file in tbl_files:
    print("Aanpassen:", tbl_file)
    multiply_tbl(tbl_file, factor)

Aantal gevonden parameterbestanden: 1
Aanpassen: /home/niels/BEP-Niels/temp_runs/sens_InfiltCapSoil_x1p25/intbl/InfiltCapSoil.tbl


In [88]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Run klaar")

365 1986-12-31 00:00:00 11876.8
730 1987-12-31 00:00:00 2224.0
1095 1988-12-30 00:00:00 3958.7
1460 1989-12-30 00:00:00 4676.8
1825 1990-12-30 00:00:00 6410.8
2190 1991-12-30 00:00:00 7189.5
2555 1992-12-29 00:00:00 1827.6
2920 1993-12-29 00:00:00 10864.1
3285 1994-12-29 00:00:00 5357.5
3650 1995-12-29 00:00:00 10540.7
Run klaar


In [89]:
model_InfiltCapSoil = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_InfiltCapSoil["date"] = model_InfiltCapSoil["date"].dt.tz_convert(None).dt.floor("D")

model_InfiltCapSoil = model_InfiltCapSoil[
    (model_InfiltCapSoil["date"] >= calibration_start) &
    (model_InfiltCapSoil["date"] <= calibration_end)
].copy()

output_file = results_dir / f"{run_name}_lobith_daily.csv"

model_InfiltCapSoil.to_csv(output_file, index=False)

print("Aantal dagen:", len(model_InfiltCapSoil))
print("Opgeslagen als:", output_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/sens_InfiltCapSoil_x1p25_lobith_daily.csv


In [90]:
comparison_InfiltCapSoil = pd.merge(
    grdc_calibration,
    model_InfiltCapSoil,
    on="date",
    how="inner"
)

scores_InfiltCapSoil = scores(comparison_InfiltCapSoil)

pd.DataFrame(
    [basis_scores_full, scores_InfiltCapSoil],
    index=["basis_full", run_name]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_InfiltCapSoil_x1p25,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38


In [91]:
check = pd.merge(
    model_basis_full,
    model_InfiltCapSoil,
    on="date",
    suffixes=("_basis", "_infilt")
)

check["verschil"] = check["Q_model_m3s_infilt"] - check["Q_model_m3s_basis"]

print("Max verschil:", check["verschil"].abs().max())
print("Gemiddeld verschil:", check["verschil"].mean())

Max verschil: 0.0
Gemiddeld verschil: 0.0


In [93]:
pet_factor = 0.90

forcing_pet_file = forcing_file.parent / "wflow_ERA5_Rhine_1986_1995_pet_x0p90.nc"

ds = xr.open_dataset(forcing_file)
ds_new = ds.copy()

ds_new["pet"] = ds_new["pet"] * pet_factor

ds_new.to_netcdf(forcing_pet_file)

ds.close()
ds_new.close()

print("Nieuwe forcing opgeslagen:")
print(forcing_pet_file)

Nieuwe forcing opgeslagen:
/home/niels/BEP-Niels/temp_runs/forcing_basisrun_1987_1995/work/diagnostic/script/wflow_ERA5_Rhine_1986_1995_pet_x0p90.nc


In [94]:
forcing_pet = WflowForcing(
    start_time=model_start,
    end_time=model_end,
    directory=str(forcing_pet_file.parent),
    netcdfinput=forcing_pet_file.name,
    Precipitation="/pr",
    EvapoTranspiration="/pet",
    Temperature="/tas",
    Inflow=None,
)

print("PET forcing gekoppeld")

PET forcing gekoppeld


In [95]:
run_name = "sens_PET_x0p90"
run_dir = temp_dir / run_name

print("Run:", run_name)
print("Runmap:", run_dir)

Run: sens_PET_x0p90
Runmap: /home/niels/BEP-Niels/temp_runs/sens_PET_x0p90


In [96]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing_pet
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))

print("PET-run voorbereid")

PET-run voorbereid


In [97]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Run klaar")

365 1986-12-31 00:00:00 12001.1
730 1987-12-31 00:00:00 2274.2
1095 1988-12-30 00:00:00 4104.1
1460 1989-12-30 00:00:00 5041.4
1825 1990-12-30 00:00:00 6808.0
2190 1991-12-30 00:00:00 7592.7
2555 1992-12-29 00:00:00 1891.6
2920 1993-12-29 00:00:00 10952.1
3285 1994-12-29 00:00:00 5772.1
3650 1995-12-29 00:00:00 10962.2
Run klaar


In [98]:
model_pet = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_pet["date"] = model_pet["date"].dt.tz_convert(None).dt.floor("D")

model_pet = model_pet[
    (model_pet["date"] >= calibration_start) &
    (model_pet["date"] <= calibration_end)
].copy()

output_file = results_dir / f"{run_name}_lobith_daily.csv"

model_pet.to_csv(output_file, index=False)

print("Aantal dagen:", len(model_pet))
print("Opgeslagen als:", output_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/sens_PET_x0p90_lobith_daily.csv


In [99]:
comparison_pet = pd.merge(
    grdc_calibration,
    model_pet,
    on="date",
    how="inner"
)

scores_pet = scores(comparison_pet)

pd.DataFrame(
    [basis_scores_full, scores_pet],
    index=["basis_full", run_name]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.10,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_PET_x0p90,987.99,2256.18,-0.95,55.32,635.29,1150,1121,-29,125,692,567,1187.2,648.40,-538.80,1310.62,879.59,-431.03


In [100]:
pet_factor = 0.95

forcing_pet_dir = temp_dir / "forcing_pet_x0p95_1987_1995"
forcing_pet_dir.mkdir(exist_ok=True)

forcing_pet_file = forcing_pet_dir / "wflow_ERA5_Rhine_1986_1995_pet_x0p95.nc"

if forcing_pet_file.exists():
    print("PET x0.95 forcing bestaat al:")
    print(forcing_pet_file)

else:
    ds = xr.open_dataset(forcing_file)
    ds_new = ds.copy()

    ds_new["pet"] = ds_new["pet"] * pet_factor

    ds_new.to_netcdf(forcing_pet_file)

    ds.close()
    ds_new.close()

    print("Nieuwe PET x0.95 forcing opgeslagen:")
    print(forcing_pet_file)

Nieuwe PET x0.95 forcing opgeslagen:
/home/niels/BEP-Niels/temp_runs/forcing_pet_x0p95_1987_1995/wflow_ERA5_Rhine_1986_1995_pet_x0p95.nc


In [101]:
forcing_pet_095 = WflowForcing(
    start_time=model_start,
    end_time=model_end,
    directory=str(forcing_pet_file.parent),
    netcdfinput=forcing_pet_file.name,
    Precipitation="/pr",
    EvapoTranspiration="/pet",
    Temperature="/tas",
    Inflow=None,
)

print("PET x0.95 forcing gekoppeld")

PET x0.95 forcing gekoppeld


In [102]:
run_name = "sens_PET_x0p95"
run_dir = temp_dir / run_name

print("Run:", run_name)
print("Runmap:", run_dir)

Run: sens_PET_x0p95
Runmap: /home/niels/BEP-Niels/temp_runs/sens_PET_x0p95


In [103]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing_pet_095
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))

print("Run voorbereid")
print(cfg_file)

Run voorbereid
/home/niels/BEP-Niels/temp_runs/sens_PET_x0p95/wflow_ewatercycle.ini


In [104]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Run klaar")

365 1986-12-31 00:00:00 11939.9
730 1987-12-31 00:00:00 2251.8
1095 1988-12-30 00:00:00 4042.6
1460 1989-12-30 00:00:00 4897.5
1825 1990-12-30 00:00:00 6673.8
2190 1991-12-30 00:00:00 7405.1
2555 1992-12-29 00:00:00 1860.8
2920 1993-12-29 00:00:00 10913.7
3285 1994-12-29 00:00:00 5593.5
3650 1995-12-29 00:00:00 10778.2
Run klaar


In [105]:
model_pet_095 = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_pet_095["date"] = model_pet_095["date"].dt.tz_convert(None).dt.floor("D")

model_pet_095 = model_pet_095[
    (model_pet_095["date"] >= calibration_start) &
    (model_pet_095["date"] <= calibration_end)
].copy()

output_file = results_dir / f"{run_name}_lobith_daily.csv"

model_pet_095.to_csv(output_file, index=False)

print("Aantal dagen:", len(model_pet_095))
print("Opgeslagen als:", output_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/sens_PET_x0p95_lobith_daily.csv


In [106]:
comparison_pet_095 = pd.merge(
    grdc_calibration,
    model_pet_095,
    on="date",
    how="inner"
)

scores_pet_095 = scores(comparison_pet_095)

pd.DataFrame(
    [basis_scores_full, scores_pet_095],
    index=["basis_full", "sens_PET_x0p95"]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.10,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_PET_x0p95,847.86,2126.10,-0.93,-33.91,598.95,1150,1211,61,125,757,632,1187.2,603.01,-584.19,1310.62,830.79,-479.83


In [107]:
pd.DataFrame(
    [basis_scores_full, scores_pet_095, scores_pet],
    index=[
        "basis_full",
        "sens_PET_x0p95",
        "sens_PET_x0p90"
    ]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.10,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_PET_x0p95,847.86,2126.10,-0.93,-33.91,598.95,1150,1211,61,125,757,632,1187.2,603.01,-584.19,1310.62,830.79,-479.83
sens_PET_x0p90,987.99,2256.18,-0.95,55.32,635.29,1150,1121,-29,125,692,567,1187.2,648.40,-538.80,1310.62,879.59,-431.03


In [108]:
run_name = "sens_N_River_x1p25"

parameter_name = "N_River.tbl"
factor = 1.25

run_dir = temp_dir / run_name

print("Run:", run_name)
print("Parameter:", parameter_name)
print("Factor:", factor)
print("Runmap:", run_dir)

Run: sens_N_River_x1p25
Parameter: N_River.tbl
Factor: 1.25
Runmap: /home/niels/BEP-Niels/temp_runs/sens_N_River_x1p25


In [109]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))
cfg_dir = Path(cfg_dir)

tbl_files = list(cfg_dir.rglob(parameter_name))

print("Aantal gevonden parameterbestanden:", len(tbl_files))

for tbl_file in tbl_files:
    print("Aanpassen:", tbl_file)
    multiply_tbl(tbl_file, factor)

Aantal gevonden parameterbestanden: 1
Aanpassen: /home/niels/BEP-Niels/temp_runs/sens_N_River_x1p25/intbl/N_River.tbl


In [110]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Run klaar")

365 1986-12-31 00:00:00 11489.1
730 1987-12-31 00:00:00 2573.9
1095 1988-12-30 00:00:00 4400.0
1460 1989-12-30 00:00:00 5228.9
1825 1990-12-30 00:00:00 6010.8
2190 1991-12-30 00:00:00 8162.0
2555 1992-12-29 00:00:00 2061.9
2920 1993-12-29 00:00:00 11560.6
3285 1994-12-29 00:00:00 4545.4
3650 1995-12-29 00:00:00 10139.6
Run klaar


In [111]:
model_N_River = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_N_River["date"] = model_N_River["date"].dt.tz_convert(None).dt.floor("D")

model_N_River = model_N_River[
    (model_N_River["date"] >= calibration_start) &
    (model_N_River["date"] <= calibration_end)
].copy()

output_file = results_dir / f"{run_name}_lobith_daily.csv"

model_N_River.to_csv(output_file, index=False)

print("Aantal dagen:", len(model_N_River))
print("Opgeslagen als:", output_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/sens_N_River_x1p25_lobith_daily.csv


In [112]:
comparison_N_River = pd.merge(
    grdc_calibration,
    model_N_River,
    on="date",
    how="inner"
)

scores_N_River = scores(comparison_N_River)

pd.DataFrame(
    [basis_scores_full, scores_N_River],
    index=["basis_full", run_name]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.10,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_N_River_x1p25,709.76,1912.76,-0.79,-90.25,551.86,1150,1265,115,125,780,655,1187.2,609.42,-577.78,1310.62,811.16,-499.46


In [113]:
check = pd.merge(
    model_basis_full,
    model_N_River,
    on="date",
    suffixes=("_basis", "_nriver")
)

check["verschil"] = check["Q_model_m3s_nriver"] - check["Q_model_m3s_basis"]

print("Max verschil:", check["verschil"].abs().max())
print("Gemiddeld verschil:", check["verschil"].mean())

Max verschil: 3221.3681640625
Gemiddeld verschil: -0.33083137858048156


In [114]:
run_name = "sens_N_x1p25"

parameter_name = "N.tbl"
factor = 1.25

run_dir = temp_dir / run_name

print("Run:", run_name)
print("Parameter:", parameter_name)
print("Factor:", factor)
print("Runmap:", run_dir)

Run: sens_N_x1p25
Parameter: N.tbl
Factor: 1.25
Runmap: /home/niels/BEP-Niels/temp_runs/sens_N_x1p25


In [115]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))
cfg_dir = Path(cfg_dir)

tbl_files = list(cfg_dir.rglob(parameter_name))

print("Aantal gevonden parameterbestanden:", len(tbl_files))

for tbl_file in tbl_files:
    print("Aanpassen:", tbl_file)
    multiply_tbl(tbl_file, factor)

Aantal gevonden parameterbestanden: 1
Aanpassen: /home/niels/BEP-Niels/temp_runs/sens_N_x1p25/intbl/N.tbl


In [116]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Run klaar")

365 1986-12-31 00:00:00 11666.1
730 1987-12-31 00:00:00 2357.7
1095 1988-12-30 00:00:00 4109.4
1460 1989-12-30 00:00:00 4892.3
1825 1990-12-30 00:00:00 6191.2
2190 1991-12-30 00:00:00 7446.8
2555 1992-12-29 00:00:00 1908.1
2920 1993-12-29 00:00:00 11266.4
3285 1994-12-29 00:00:00 5005.1
3650 1995-12-29 00:00:00 10430.9
Run klaar


In [117]:
model_N = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_N["date"] = model_N["date"].dt.tz_convert(None).dt.floor("D")

model_N = model_N[
    (model_N["date"] >= calibration_start) &
    (model_N["date"] <= calibration_end)
].copy()

output_file = results_dir / f"{run_name}_lobith_daily.csv"

model_N.to_csv(output_file, index=False)

print("Aantal dagen:", len(model_N))
print("Opgeslagen als:", output_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/sens_N_x1p25_lobith_daily.csv


In [118]:
comparison_N = pd.merge(
    grdc_calibration,
    model_N,
    on="date",
    how="inner"
)

scores_N = scores(comparison_N)

pd.DataFrame(
    [basis_scores_full, scores_N],
    index=["basis_full", run_name]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.10,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_N_x1p25,709.75,1949.19,-0.81,-104.22,558.28,1150,1272,122,125,795,670,1187.2,600.21,-586.99,1310.62,807.94,-502.68


In [119]:
check = pd.merge(
    model_basis_full,
    model_N,
    on="date",
    suffixes=("_basis", "_n")
)

check["verschil"] = check["Q_model_m3s_n"] - check["Q_model_m3s_basis"]

print("Max verschil:", check["verschil"].abs().max())
print("Gemiddeld verschil:", check["verschil"].mean())

Max verschil: 1178.3056640625
Gemiddeld verschil: -0.3430404222080473


In [120]:
run_name = "sens_N_Floodplain_x1p25"

parameter_name = "N_Floodplain.tbl"
factor = 1.25

run_dir = temp_dir / run_name

print("Run:", run_name)
print("Parameter:", parameter_name)
print("Factor:", factor)
print("Runmap:", run_dir)

Run: sens_N_Floodplain_x1p25
Parameter: N_Floodplain.tbl
Factor: 1.25
Runmap: /home/niels/BEP-Niels/temp_runs/sens_N_Floodplain_x1p25


In [122]:
if run_dir.exists():
    shutil.rmtree(run_dir)

model = Wflow(
    parameter_set=parameter_set,
    forcing=forcing
)

cfg_file, cfg_dir = model.setup(cfg_dir=str(run_dir))
cfg_dir = Path(cfg_dir)

tbl_files = list(cfg_dir.rglob(parameter_name))

print("Aantal gevonden parameterbestanden:", len(tbl_files))

for tbl_file in tbl_files:
    print("Aanpassen:", tbl_file)
    multiply_tbl(tbl_file, factor)

Aantal gevonden parameterbestanden: 1
Aanpassen: /home/niels/BEP-Niels/temp_runs/sens_N_Floodplain_x1p25/intbl/N_Floodplain.tbl


In [123]:
model.initialize(cfg_file)

dates = []
q_values = []

step = 0

while model.time < model.end_time:
    model.update()

    q = model.get_value_at_coords(
        "RiverRunoff",
        lat=[lat_lobith],
        lon=[lon_lobith]
    )[0]

    dates.append(model.time_as_datetime)
    q_values.append(float(q))

    step += 1

    if step % 365 == 0:
        print(step, model.time_as_datetime, round(float(q), 1))

model.finalize()

print("Run klaar")

365 1986-12-31 00:00:00 11876.8
730 1987-12-31 00:00:00 2224.0
1095 1988-12-30 00:00:00 3958.7
1460 1989-12-30 00:00:00 4676.8
1825 1990-12-30 00:00:00 6410.8
2190 1991-12-30 00:00:00 7189.5
2555 1992-12-29 00:00:00 1827.6
2920 1993-12-29 00:00:00 10864.1
3285 1994-12-29 00:00:00 5357.5
3650 1995-12-29 00:00:00 10540.7
Run klaar


In [124]:
model_N_Floodplain = pd.DataFrame({
    "date": pd.to_datetime(dates, utc=True),
    "Q_model_m3s": q_values
})

model_N_Floodplain["date"] = model_N_Floodplain["date"].dt.tz_convert(None).dt.floor("D")

model_N_Floodplain = model_N_Floodplain[
    (model_N_Floodplain["date"] >= calibration_start) &
    (model_N_Floodplain["date"] <= calibration_end)
].copy()

output_file = results_dir / f"{run_name}_lobith_daily.csv"

model_N_Floodplain.to_csv(output_file, index=False)

print("Aantal dagen:", len(model_N_Floodplain))
print("Opgeslagen als:", output_file)

Aantal dagen: 3287
Opgeslagen als: /home/niels/BEP-Niels/results/sens_N_Floodplain_x1p25_lobith_daily.csv


In [125]:
comparison_N_Floodplain = pd.merge(
    grdc_calibration,
    model_N_Floodplain,
    on="date",
    how="inner"
)

scores_N_Floodplain = scores(comparison_N_Floodplain)

pd.DataFrame(
    [basis_scores_full, scores_N_Floodplain],
    index=["basis_full", run_name]
).round(2)

,bias,rmse,log_nse,bias_low_1600,mae_low_1600,grdc_days_below_1600,model_days_below_1600,diff_days_below_1600,grdc_days_below_1020,model_days_below_1020,diff_days_below_1020,grdc_p10,model_p10,diff_p10,grdc_p17,model_p17,diff_p17
basis_full,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38
sens_N_Floodplain_x1p25,710.1,1998.63,-0.94,-116.22,575.02,1150,1291,141,125,812,687,1187.2,572.26,-614.94,1310.62,779.24,-531.38


In [1]:
def multiply_tbl_max1(tbl_file, factor):
    lines = Path(tbl_file).read_text().splitlines()
    new_lines = []

    for line in lines:
        parts = line.split()

        try:
            value = float(parts[-1])
            new_value = value * factor
            new_value = min(new_value, 1.0)
            parts[-1] = str(new_value)
            new_lines.append(" ".join(parts))
        except:
            new_lines.append(line)

    Path(tbl_file).write_text("\n".join(new_lines))

In [7]:
run_name = "sens_CanopyGapFraction_x1p25"

parameter_name = "CanopyGapFraction.tbl"
factor = 1.25

run_dir = temp_dir / run_name

print("Run:", run_name)
print("Parameter:", parameter_name)
print("Factor:", factor)
print("Runmap:", run_dir)

Run: sens_CanopyGapFraction_x1p25
Parameter: CanopyGapFraction.tbl
Factor: 1.25
Runmap: /home/niels/BEP-Niels/temp_runs/sens_CanopyGapFraction_x1p25
